# Phase 5 & 6: Model Training, Cross-Validation & Model Selection

**Project**: AI-Based Sales Conversion Probability Prediction  
**Dataset**: UCI Online Shoppers Purchasing Intention Dataset  
**Notebook Author Role**: Phase 5 — Model Selection via Stratified CV | Phase 6 — Final Model Locking  
**Status**: Documentation notebook. Training was executed via `scratch/run_phase5.py` and `scratch/execute_phase6.py`. This notebook presents the methodology, decisions, and previously recorded results without re-executing training.

---

> **Important**: This notebook does NOT re-execute model training or cross-validation experiments.  
> All results, metrics, and decisions below are sourced from previously saved artifacts:  
> - `reports/cv_experiment_results.csv` (168 CV experiments, run via `scratch/run_phase5.py`)  
> - `models/model_metadata.json` (locked model specification and performance)  
> - `reports/model_results.md` (Phase 5 and Phase 6 combined benchmark report)  
> - Direct joblib inspection of `models/best_lead_scoring_model.joblib`  
>
> Rerunning the CV matrix would reproduce equivalent results (given the same `random_state=42`) but takes significant time. Results here are presented as previously recorded.

## 1. Training Methodology Overview

The model training pipeline follows a strict **holdout isolation** protocol:

```
Raw Dataset (12,330 rows)
    │
    ▼ Pre-split deduplication (125 exact duplicates removed)
Unique Sessions (12,205 rows)
    │
    ▼ Stratified train/test split (80/20, random_state=42)
    ├── X_train / y_train (9,764 rows) ← Used for ALL CV experiments
    └── X_test  / y_test  (2,441 rows) ← Held out until Phase 6 final evaluation ONLY
```

**Key Principles Applied:**
1. All feature scaling, encoding, and imputation are fitted **inside each CV fold** to prevent data leakage.
2. Model selection is performed **exclusively** using `X_train` via Stratified K-Fold Cross-Validation.
3. The holdout test set `X_test` was accessed **exactly once** in Phase 6 after the model was locked.
4. No hyperparameters were tuned using test-set information.

## 2. Preprocessing and Pipeline Architecture

The full prediction pipeline is:

```
Raw Input (17 columns)
    │
    ▼ FeatureEngineeringTransformer (src/feature_engineering.py)
    │   ├── include_pagevalues = True  → keeps PageValues
    │   ├── page_count_config = 'original' → keeps Administrative, Informational, ProductRelated
    │   └── duration_config = 'original'  → keeps *_Duration columns (no aggregation)
    │
    ▼ ColumnTransformer Preprocessor (src/data_preprocessing.py)
    │   ├── num_raw  → Median Imputer only (no scaling — appropriate for RF)
    │   │   └── [SpecialDay, BounceRates, ExitRates, Administrative, Informational,
    │   │        ProductRelated, Administrative_Duration, Informational_Duration,
    │   │        ProductRelated_Duration, PageValues]  (10 features)
    │   ├── cat  → OneHotEncoder (handle_unknown='ignore')
    │   │   └── [Month, VisitorType, OperatingSystems, Browser, Region, TrafficType]  (→62 OHE features)
    │   └── bool → Integer cast (0/1)
    │       └── [Weekend]  (1 feature)
    │
    ▼ RandomForestClassifier
       └── 73 total input features
```

**Why no scaling for Random Forest?**  
Random Forests use decision tree splits on individual feature thresholds. They are **invariant to monotonic feature transformations** such as StandardScaler. Scaling is only needed for distance-based or gradient-based algorithms (e.g., Logistic Regression, SVM, neural networks).

## 3. Stratified Cross-Validation Methodology

### Why Stratified K-Fold?

The dataset has an **imbalanced class distribution** (~15.6% conversion rate). Regular K-Fold splits could produce folds with very different positive-class rates, destabilizing fold-level metrics. **Stratified K-Fold** preserves the class ratio in each training and validation fold.

### Configuration Used
```python
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

- **5 folds**: Each fold uses ~7,811 training rows and ~1,953 validation rows
- `shuffle=True`: Randomises row order before fold assignment
- `random_state=42`: Ensures reproducible fold assignments

### Fold Isolation Protocol

Within each fold:
```
X_tr (fold train) → FeatureEngineer.fit_transform() → Preprocessor.fit_transform() → Classifier.fit()
X_val (fold val)  → FeatureEngineer.transform()     → Preprocessor.transform()     → Classifier.predict_proba()
```
The preprocessor (imputer, encoder) is **never fitted on validation data**.

## 4. CV Experiment Matrix — Configuration Space

The full experiment matrix (`scratch/run_phase5.py` → `src/train.py:run_experiment_matrix()`) evaluated **168 experiments** covering:

| Dimension | Values Evaluated |
|---|---|
| Algorithm | `RandomForestClassifier`, `LogisticRegression` |
| PageValues inclusion | `True`, `False` |
| Page count configuration | `'original'` (component counts), `'total'` (aggregated) |
| Duration configuration | `'original'` (component durations), `'total'` (aggregated) |
| Class imbalance strategy | `'none'`, `'balanced'`, `'smote'` |
| RF hyperparameter sets | `{n_est=100, max_depth=10, min_leaf=2}`, `{n_est=200, max_depth=15, min_leaf=2}`, `{n_est=200, max_depth=None, min_leaf=5}` |
| LR hyperparameter sets | C ∈ {0.01, 0.1, 1.0, 10.0} |

Total: 2 × 2 × 4 × 3 = 48 RF + 2 × 4 × 3 × 4 = 96 LR = 144... plus near-tie focused runs = **168 rows** in `reports/cv_experiment_results.csv`.

Full results are stored in `reports/cv_experiment_results.csv`.

In [ ]:
# Load and display top CV experiment results (read-only — no retraining)
import pandas as pd
import sys
sys.path.insert(0, '..')

df_results = pd.read_csv('../reports/cv_experiment_results.csv')
print(f"Total CV experiments recorded: {len(df_results)}")
print(f"Columns: {list(df_results.columns)}")
print()

# Top 10 by ROC-AUC
top10_display = df_results.head(10)[[
    'model_type', 'include_pagevalues', 'page_count_config', 'duration_config',
    'imbalance_strategy', 'model_kwargs', 'mean_roc_auc', 'std_roc_auc',
    'mean_pr_auc', 'mean_brier'
]]
top10_display

## 5. Class Imbalance Strategy Comparison

Three strategies were evaluated for handling the ~15.6% positive-class imbalance:

| Strategy | Mechanism | Trade-off |
|---|---|---|
| `class_weight=None` | No adjustment — model learns from raw class distribution | Best calibration, highest precision |
| `class_weight='balanced'` | Loss function weights inversely proportional to class frequency | Higher recall, worse calibration (inflated probabilities) |
| `SMOTE` | Synthetic minority oversampling of training fold | Higher recall, computational overhead, mild calibration degradation |

### Results for Selected Configuration (`n_estimators=200, max_depth=None, min_samples_leaf=5`, Config A, `include_pagevalues=True`)

*(Source: `reports/cv_experiment_results.csv`, verified rows 2, 3, 11)*

| Imbalance Strategy | Mean ROC-AUC ± SD | Mean PR-AUC ± SD | Mean Brier ± SD | Mean Precision | Mean Recall |
|---|---|---|---|---|---|
| `balanced` | 0.9279 ± 0.0040 | 0.7294 ± 0.0195 | 0.0874 ± 0.0023 | 0.5951 | 0.7844 |
| `smote` | 0.9274 ± 0.0055 | 0.7213 ± 0.0267 | 0.0745 ± 0.0028 | 0.6882 | 0.6573 |
| **`none` (selected ✅)** | **0.9267 ± 0.0059** | **0.7474 ± 0.0199** | **0.0739 ± 0.0021** | **0.8077** | 0.4948 |

**Source**: `cv_experiment_results.csv` rows 2 (`balanced`), 3 (`smote`), 11 (`none`). Verified.

### Key Insight — Why `class_weight=None` Was Selected

The ROC-AUC values for `balanced` (0.9279) and `none` (0.9267) are in an effective **near-tie** (Δ = +0.0012).  
A focused paired t-test across the 5 CV folds (run in `scratch/near_tie_analysis.py`) yielded **p = 0.4602 > 0.05** (reported; raw fold values not persisted to disk).

Applying the predefined secondary-metric tie-breaking hierarchy:
- `none` achieves **superior PR-AUC** (0.7474 vs 0.7294 for `balanced`)
- `none` achieves a **lower Brier score** (0.0739 vs 0.0874 for `balanced`), indicating better probability calibration
- `balanced` artificially shifts predicted probabilities upward, degrading calibration

**Decision**: `class_weight=None` selected per predefined hierarchy: *prefer simpler, better-calibrated model in near-ties*.

In [ ]:
# Display Class Imbalance Strategy comparison for the selected RF configuration
mask_selected_config = (
    (df_results['model_type'] == 'random_forest') &
    (df_results['include_pagevalues'] == True) &
    (df_results['page_count_config'] == 'original') &
    (df_results['duration_config'] == 'original') &
    (df_results['model_kwargs'].str.contains("'max_depth': None")) &
    (df_results['model_kwargs'].str.contains("'min_samples_leaf': 5"))
)
imb_comparison = df_results[mask_selected_config][[
    'imbalance_strategy', 'mean_roc_auc', 'std_roc_auc',
    'mean_pr_auc', 'std_pr_auc', 'mean_brier', 'std_brier',
    'mean_precision', 'mean_recall'
]].sort_values('imbalance_strategy')
print("Class Imbalance Strategy Comparison (Config A, max_depth=None, min_samples_leaf=5):")
imb_comparison

## 6. Hyperparameter Configuration Comparison

Three Random Forest hyperparameter sets were evaluated for `class_weight=None`, Config A, `include_pagevalues=True`:

| Hyperparameter Set | Mean ROC-AUC ± SD | Mean PR-AUC | Mean Brier | CSV Source |
|---|---|---|---|---|
| `n_est=100, max_depth=10, min_leaf=2` | 0.9240 ± 0.0056 | 0.7462 | 0.0754 | Row 22 |
| `n_est=200, max_depth=15, min_leaf=2` | 0.9254 ± 0.0054 | 0.7470 | 0.0731 | Row 19 |
| **`n_est=200, max_depth=None, min_leaf=5` (selected ✅)** | **0.9267 ± 0.0059** | **0.7474** | **0.0739** | Row 11 |

**Source**: `cv_experiment_results.csv`. Verified.

> **Note on `max_depth=15, min_leaf=2`**: This configuration appeared in an intermediate report template as the "locked model" but was **never saved as the production artifact**. The saved model (`models/best_lead_scoring_model.joblib`) uses `max_depth=None, min_samples_leaf=5`, verified by direct joblib inspection. See `reports/model_results.md` for the resolution of this audit finding.

In [ ]:
# Display hyperparameter comparison for class_weight=None, Config A, include_pagevalues=True
mask_none = (
    (df_results['model_type'] == 'random_forest') &
    (df_results['include_pagevalues'] == True) &
    (df_results['page_count_config'] == 'original') &
    (df_results['duration_config'] == 'original') &
    (df_results['imbalance_strategy'] == 'none')
)
hp_comparison = df_results[mask_none][[
    'model_kwargs', 'mean_roc_auc', 'std_roc_auc', 'mean_pr_auc', 'mean_brier'
]].sort_values('mean_roc_auc', ascending=False)
print("RF Hyperparameter Comparison (class_weight=None, Config A, PageValues=True):")
hp_comparison

## 7. PageValues Sensitivity Analysis

To assess dependency on `PageValues`, we compared the top-performing model configuration with and without it.

*(Source: `cv_experiment_results.csv`. Rows identified by `include_pagevalues` column.)*

| Configuration | Mean CV ROC-AUC | Mean CV PR-AUC | Mean CV Brier |
|---|---|---|---|
| **Model Set A — With `PageValues`** (`class_weight=None`, original config) | 0.9267 | 0.7474 | 0.0739 |
| **Model Set B — Without `PageValues`** (best RF, `balanced`) | 0.7773 | 0.3887 | 0.1462 |

> **Source clarification**: 'Without PageValues' row reflects `random_forest, False, original, original, balanced, {n_est=200, max_depth=None, min_leaf=5}` (CSV row 86, ROC-AUC=0.7773). The best without-PageValues RF under `class_weight=None` is CSV row 90 (ROC-AUC=0.7757).

**PageValues Availability Note**: `PageValues` reflects accumulated Google Analytics session interaction data, available from the browsing session at prediction time. The project treats this as a valid prediction-time feature under the **session-level prediction assumption**. However:
- If the prediction must occur at session **start** (before any pageview accumulation), `PageValues` may not yet be non-zero.
- The sensitivity experiment (Model Set B) demonstrates the model retains meaningful signal (~0.776 ROC-AUC) without PageValues.
- Teams deploying this model should confirm that PageValues is available in their real-time event pipeline before the prediction API call.

In [ ]:
# PageValues sensitivity comparison from CSV
pv_true = df_results[
    (df_results['include_pagevalues'] == True) &
    (df_results['model_type'] == 'random_forest')
].head(1)[['include_pagevalues','imbalance_strategy','model_kwargs','mean_roc_auc','mean_pr_auc','mean_brier']]

pv_false = df_results[
    (df_results['include_pagevalues'] == False) &
    (df_results['model_type'] == 'random_forest')
].head(3)[['include_pagevalues','imbalance_strategy','model_kwargs','mean_roc_auc','mean_pr_auc','mean_brier']]

print("Top model WITH PageValues:")
print(pv_true.to_string(index=False))
print()
print("Top 3 models WITHOUT PageValues:")
print(pv_false.to_string(index=False))

## 8. Model Selection Decision Summary

**Selection Hierarchy (predefined before any test-set access):**
1. **Primary**: Maximise mean CV ROC-AUC
2. **Secondary** (applied in near-ties): PR-AUC, Brier score (calibration), fold-to-fold stability
3. **Tie-breaking rule**: Prefer simpler (fewer hyperparameter adjustments) and better-calibrated model

**Result**:

| Decision Point | Choice Made | Rationale |
|---|---|---|
| Algorithm | Random Forest | ROC-AUC 0.9267 vs best LR 0.9003 |
| PageValues | Included (Config A) | ROC-AUC 0.9267 vs 0.7773 without |
| Imbalance strategy | `class_weight=None` | Near-tie with `balanced` (p=0.4602), but superior PR-AUC & Brier |
| Hyperparameters | `n_est=200, max_depth=None, min_leaf=5` | Highest ROC-AUC among three RF sets |
| Feature config | `original` pages + `original` durations | Best CV performance combination |

## 9. Final Locked Model Configuration

```python
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline

# LOCKED — Do not modify
rf_kwargs = {
    "n_estimators": 200,
    "max_depth": None,       # Fully grown trees
    "min_samples_leaf": 5,   # Minimum 5 samples per leaf
    "class_weight": None,    # No class reweighting
    "random_state": 42
}

pipeline = ImbPipeline([
    ("feature_engineer", FeatureEngineeringTransformer(
        include_pagevalues=True,
        page_count_config='original',
        duration_config='original'
    )),
    ("preprocessor", build_preprocessing_pipeline(model_type='random_forest', ...)),
    ("classifier", RandomForestClassifier(**rf_kwargs))
])
```

**Serialized artifact**: `models/best_lead_scoring_model.joblib`  
**Metadata**: `models/model_metadata.json`  
**Training execution script**: `scratch/execute_phase6.py`

In [ ]:
# Verify saved model configuration directly from joblib (read-only inspection)
import joblib

pipeline = joblib.load('../models/best_lead_scoring_model.joblib')
clf = pipeline.named_steps['classifier']
fe  = pipeline.named_steps['feature_engineer']

print("=== Saved Model: Classifier Hyperparameters ===")
for k in ['n_estimators', 'max_depth', 'min_samples_leaf', 'class_weight', 'random_state']:
    print(f"  {k}: {clf.get_params()[k]}")

print()
print("=== Saved Model: Feature Engineer Config ===")
print(f"  include_pagevalues:             {fe.include_pagevalues}")
print(f"  page_count_config:              {fe.page_count_config}")
print(f"  duration_config:                {fe.duration_config}")
print(f"  include_product_page_ratio:     {fe.include_product_page_ratio}")
print(f"  include_avg_time_per_product_page: {fe.include_avg_time_per_product_page}")

print()
print("=== Pipeline Steps ===")
print([name for name, _ in pipeline.steps])

print()
print(f"=== Fitted Model Evidence ===")
print(f"  Number of trees fitted: {len(clf.estimators_)}")
print(f"  Input features to classifier: {clf.n_features_in_}")

## 10. Phase 6 — Final Holdout Evaluation (Previously Recorded)

After model selection was finalised from CV results, the locked pipeline was **fitted once on the full `X_train`** and then evaluated **exactly once** on the untouched `X_test`.

**Source**: `models/model_metadata.json` (written by `scratch/execute_phase6.py` during the single Phase 6 execution).

### Cross-Validation vs Holdout Test Comparison

| Metric | 5-Fold CV on X_train (Mean ± SD) | Holdout Test X_test (Reported) |
|---|---|---|
| **ROC-AUC** | **0.9267 ± 0.0059** | **0.9293** |
| **PR-AUC** | **0.7474 ± 0.0199** | **0.7458** |
| **Brier Score** | **0.0739 ± 0.0021** | **0.0722** |
| Accuracy | 0.9026 ± 0.0031 | 0.9033 |
| Precision (0.50) | 0.8077 ± 0.0210 | 0.7645 |
| Recall (0.50) | 0.4948 ± 0.0250 | 0.5524 |
| F1-Score (0.50) | 0.6133 ± 0.0230 | 0.6413 |

> **Important label**: CV metrics are VERIFIED from `cv_experiment_results.csv`. Holdout test metrics are REPORTED from `models/model_metadata.json` (generated during Phase 6 execution). The holdout test was not rerun during this audit.

### Holdout Confusion Matrix (at 0.50 threshold, Reported)

```
Predicted:    Non-Converging  Converting
Actual Non-Converging:  1994          65     (TN=1994, FP=65)
Actual Converting:       171         211     (FN=171,  TP=211)
```

- Positive class count: 382 / 2,441 = **15.65%** conversion rate (consistent with training: 15.63%)

### Generalization Assessment

The holdout ROC-AUC (0.9293) is within **0.44 standard deviations** of the CV mean (0.9267 ± 0.0059). This indicates **excellent generalization** with no meaningful overfitting.

In [ ]:
# Load and display model metadata (read-only)
import json

with open('../models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

print("=== CV Performance (from metadata) ===")
for k, v in metadata['cv_performance'].items():
    print(f"  {k}: {v}")

print()
print("=== Holdout Test Performance (from metadata — REPORTED) ===")
for k, v in metadata['holdout_test_performance'].items():
    if k != 'confusion_matrix':
        print(f"  {k}: {v}")
    else:
        print(f"  confusion_matrix: TN={v['tn']}, FP={v['fp']}, FN={v['fn']}, TP={v['tp']}")

## 11. Evaluation Methodology and Limitations

### Metric Choices

| Metric | Role | Rationale |
|---|---|---|
| **ROC-AUC** | Primary | Threshold-independent discrimination; robust to class imbalance; suitable for ranking leads |
| **PR-AUC** | Secondary | Focuses on positive class (converters); better than ROC-AUC under heavy imbalance |
| **Brier Score** | Secondary | Measures probability calibration; lower = better-calibrated |
| Precision/Recall/F1 | Diagnostic | Threshold-dependent; shown at default 0.50 as an illustrative baseline |

### Known Limitations

1. **Threshold**: The 0.50 classification boundary is a default, not an optimised business threshold. Real deployment requires cost-sensitive analysis (false positive cost vs false negative cost).
2. **PageValues availability**: Requires confirmation that PageValues is populated in the real-time event stream at the moment of prediction.
3. **Near-tie p-value**: The p = 0.4602 value (Balanced vs None, paired t-test) is reported from `near_tie_analysis.py` execution. Raw fold-level values were not persisted to disk and cannot be independently recomputed without re-running the CV.
4. **Temporal generalization**: The dataset is a cross-sectional snapshot. Model performance on future sessions under distribution shift is unknown.
5. **Causation vs correlation**: High SHAP values for a feature indicate strong predictive association, not causal influence on purchase decisions.

### Reproducibility

To reproduce the full training run:
```bash
# Phase 5: Full CV experiment matrix (takes ~15-20 minutes)
python scratch/run_phase5.py

# Phase 6: Fit locked model on X_train, evaluate on X_test, save artifacts
python scratch/execute_phase6.py
```

With `random_state=42` and identical library versions (see `requirements.txt`), results should reproduce closely. Minor numerical differences may arise from floating-point non-determinism in parallel tree building.